In [ ]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

In [ ]:
!pip install pydantic-ai-slim openai pandas numpy


In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

llm_30b = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

In [ ]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score


user_df = pd.DataFrame({
    "user_id": [f"U{i}" for i in range(1000)],
    "account_age_days": np.random.randint(10, 1000, 1000)
})


device_df = pd.DataFrame({
    "device_id": [f"D{i}" for i in range(1000)],
    "user_id": np.random.choice(user_df["user_id"], 1000),
    "device_consistency": np.random.uniform(0.7, 1.0, 1000),
    "is_new_device": np.random.choice([0, 1], 1000)
})



behavior_df = pd.DataFrame({
    "user_id": user_df["user_id"],
    "behavior_score": np.random.uniform(0.6, 1.0, len(user_df))
})


events_df = pd.DataFrame({
    "event_id": [f"E{i}" for i in range(5000)],
    "user_id": np.random.choice(user_df["user_id"], 5000),
    "device_id": np.random.choice(device_df["device_id"], 5000),
    "sim_changed_recently": np.random.choice([0, 1], 5000),
    "geo_distance_from_last_login": np.random.uniform(0, 50, 5000)
})


def generate_labels(events_df):
    labels = []

    for _, row in events_df.iterrows():
        risk = (
            0.3 * row["sim_changed_recently"] +
            0.2 * (row["geo_distance_from_last_login"] > 20)
        )
        labels.append(1 if risk > 0.4 else 0)

    return pd.DataFrame({
        "event_id": events_df["event_id"],
        "is_fraud": labels
    })

labels_df = generate_labels(events_df)


df = events_df.merge(labels_df, on="event_id")
df = df.merge(device_df, on=["user_id", "device_id"], how="left")
df = df.merge(behavior_df, on="user_id", how="left")



features = [
    "device_consistency",
    "is_new_device",
    "behavior_score",
    "sim_changed_recently",
    "geo_distance_from_last_login"
]

X = df[features]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

ml_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

ml_model.fit(X_train, y_train)

y_prob = ml_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)  # 🔥 tuned threshold

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

tn, fp, fn, tp = cm.ravel()

print(f"""
False Negatives (MOST IMPORTANT): {fn}
Recall: {recall_score(y_test, y_pred)}
ROC-AUC: {roc_auc_score(y_test, y_prob)}
""")



y_prob = ml_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)  # 🔥 tuned threshold

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

tn, fp, fn, tp = cm.ravel()

print(f"""
False Negatives (MOST IMPORTANT): {fn}
Recall: {recall_score(y_test, y_pred)}
ROC-AUC: {roc_auc_score(y_test, y_prob)}
""")


def context_agent(user_id, device_id):
    device = device_df[
        (device_df["user_id"] == user_id) &
        (device_df["device_id"] == device_id)
    ].iloc[0]

    return {
        "device_consistency": device["device_consistency"],
        "is_new_device": device["is_new_device"]
    }



def telecom_agent(event):
    return {
        "sim_changed_recently": event["sim_changed_recently"],
        "geo_distance_from_last_login": event["geo_distance_from_last_login"]
    }


def behavior_agent(user_id):
    row = behavior_df[behavior_df["user_id"] == user_id].iloc[0]

    return {
        "behavior_score": row["behavior_score"]
    }



def risk_agent(features):

    df_input = pd.DataFrame([features])
    prob = ml_model.predict_proba(df_input)[0][1]

    # 🔥 Rule overrides
    if features["sim_changed_recently"] == 1:
        prob = max(prob, 0.8)

    if features["is_new_device"] == 1:
        prob = max(prob, 0.5)

    return {
        "risk_score": round(prob, 2),
        "risk_level": "high" if prob > 0.7 else "medium" if prob > 0.3 else "low"
    }



from pydantic_ai import Agent

decision_agent = Agent(
    model=llm_30b,
    system_prompt="""
    Decide authentication:

    low → no OTP
    medium → step-up
    high → OTP

    Return JSON with decision + reason.
    """
)

async def run_auth_flow(event):

    user_id = event["user_id"]
    device_id = event["device_id"]

    features = {
        **context_agent(user_id, device_id),
        **telecom_agent(event),
        **behavior_agent(user_id)
    }

    risk = risk_agent(features)

    prompt = f"""
    Risk Score: {risk['risk_score']}
    Risk Level: {risk['risk_level']}
    Features: {features}
    """

    decision = await decision_agent.run(prompt)

    return {
        "features": features,
        "risk": risk,
        "decision": decision.output
    }

In [ ]:
sample_event = events_df.iloc[0]

result = await run_auth_flow(sample_event)

print("FINAL OUTPUT:\n", result)